# Voltway data agent lab

Run `./scripts/setup.sh` first (Docker stack, DataHub, catalogue). Then work through the cells: check the stack,
look at what each MCP server offers, and ask the agent questions while watching every step it takes.

In [ ]:
import subprocess, sys
print(subprocess.run([sys.executable, "scripts/check_stack.py"], capture_output=True, text=True).stdout)

## Start the agent
This opens one session per MCP server (DataHub, Cube, DuckDB, Postgres) and keeps them open.

In [ ]:
from data_agent import DataAgent

agent = await DataAgent(checkpointer="memory").start()  # "postgres" to save conversations

by_server = {}
for tool_name, server in agent.sessions.server_of.items():
    by_server.setdefault(server, []).append(tool_name)
for server, names in by_server.items():
    print(f"{server:>9}: {', '.join(sorted(names))}")

## A helper that shows each step

In [ ]:
import json

async def ask(question: str, thread_id: str = "lab"):
    async for event in agent.stream(question, thread_id=thread_id):
        if event[0] == "route":
            print(f"🧭 route: {event[1]} ({event[2]})")
        elif event[0] == "tool_call":
            _, name, args, _ = event
            print(f"🔧 {name}: {args.get('sql') or json.dumps(args)[:300]}")
        elif event[0] == "tool_result":
            _, name, text, is_error, _ = event
            print(f"   {'❌' if is_error else '↳'} {text[:300]}")
        else:
            print("\n" + event[1])

## 1. A governed metric: answered from the Cube semantic layer

In [ ]:
await ask("Which vehicle model is the most energy efficient, in kWh per 100 km?")

## 2. A follow-up in the same conversation (the checkpointer remembers it)

In [ ]:
await ask("And how does that compare between depot regions?")

## 3. Federated SQL: Postgres tables + the energy_prices file, through DuckDB

In [ ]:
await ask("Did we pay the depot tariff for depot charging in March? Compare what we paid per kWh with the tariff, by week.", thread_id="lab-2")

## 4. The catalogue: what data is there, and what is personal data?

In [ ]:
await ask("What data do we have about drivers, and which columns are personal data?", thread_id="lab-3")

## 5. Guardrails: a write is refused before it reaches the database

In [ ]:
await ask("Delete all trips from January, they were test data.", thread_id="lab-4")

In [ ]:
await agent.stop()